# Notebook 01 v4 — historical issuer and price recovery

**Run after your completed v3 reconciliation.** This is a focused recovery notebook, not another universe-wide download. It prioritizes the ten largest 2016–2019 price gaps plus Reynolds American and legacy Allergan. Original v2/v3 files and the 2020+ model baseline stay unchanged.

The revision uses filing-based issuer CIKs, company-name search, CIK profiles, and a bounded delisted-directory scan. It follows discovered vendor symbols only as **unapproved retrieval candidates**. A matching CIK identifies an issuer, not a particular share class or valid historical price series. Present-day metadata never becomes a feature.

It also corrects v3's full-EOD schema check: `close` is valid diagnostic output; absent `adjClose` is not an HTTP failure. RAI receives a conditional dividend-based reconstruction and overlap diagnostics, with no automatic approval or replacement of labels.

**Run All** from the project's `Notebooks` folder. The paths below are pinned to your completed runs. Re-running reuses caches, including empty successful responses. Up to 600 new HTTP attempts / one hour; partial progress is reported and resumable.

In [1]:
from pathlib import Path
import os, json, hashlib, time, zipfile, gzip, re
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
from IPython.display import display
# If needed: %pip install pandas numpy requests python-dotenv pyarrow
HERE=Path.cwd().resolve()
PROJECT_ROOT=HERE.parent if HERE.name.lower()=='notebooks' else HERE
DATA=PROJECT_ROOT/'data'
SOURCE_RUN=DATA/'interim/audits/nb01_history_v3/20261020T140634856943Z'
V2_RUN=DATA/'interim/audits/nb01_history_v2/20261019T230406622432Z'
OLD_CACHES=[DATA/'raw/history_expansion/historical_expansion_v2_20261019',
            DATA/'raw/history_expansion/historical_reconciliation_v3']
NEW_CACHE=DATA/'raw/history_expansion/historical_issuer_recovery_v4'
VOO_CALENDAR=DATA/'raw/market/VOO_dividend_adjusted_2010_plus.parquet'
RUN_NETWORK=True
MAX_NEW_HTTP_ATTEMPTS=600
MAX_RUN_SECONDS=3600
REQUEST_INTERVAL_SECONDS=0.35
MAX_DELISTED_PAGES=20
MAX_PROFILE_LEADS_PER_ISSUER=5
MAX_EXTRA_SYMBOLS_PER_ISSUER=3
BASE_URL='https://financialmodelingprep.com/stable'
for folder in [PROJECT_ROOT,HERE]:
    for name in ['.env','env']:
        if (folder/name).is_file():load_dotenv(folder/name,override=False)
FMP_KEY=os.getenv('FMP_API_KEY') or os.getenv('FMP_KEY') or ''
for p in [SOURCE_RUN,V2_RUN,VOO_CALENDAR,*OLD_CACHES]:
    if not p.exists():raise FileNotFoundError(f'Check PROJECT_ROOT and source paths: {p}')
RUN_ID=pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR=DATA/'interim/audits/nb01_history_v4'/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=False)
print('Output:',RUN_DIR)

Output: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb01_history_v4/20261020T142058312830Z


In [2]:
def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), default=str)

def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=str))
    temporary.replace(path)

class FetchStop(Exception):
    pass

class FMPArchive:
    """Cache identity includes every public request parameter; never stores credentials."""
    def __init__(self, root, key, budget, seconds, network=True):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.key, self.budget, self.seconds, self.network = key, budget, seconds, network
        self.calls, self.started, self.last_call = 0, time.monotonic(), 0
        self.session = requests.Session()
        self.log = []

    def request_id(self, endpoint, params):
        return hashlib.sha256(canonical_json({'endpoint': endpoint, 'params': params}).encode()).hexdigest()

    def get(self, endpoint, params):
        request_id = self.request_id(endpoint, params)
        path = self.root / (request_id + '.json')
        if path.exists():
            record = json.loads(path.read_text())
            if record['endpoint'] != endpoint or record['params'] != params:
                raise ValueError('Cache request mismatch')
            if hashlib.sha256(canonical_json(record['data']).encode()).hexdigest() != record['payload_sha256']:
                raise ValueError('Cache payload checksum mismatch')
            return record['data'], 'cached', request_id
        if not self.network:
            raise FetchStop('network_disabled_cache_miss')
        if not self.key:
            raise FetchStop('missing_FMP_API_KEY')
        for attempt in range(3):
            if self.calls >= self.budget or time.monotonic() - self.started >= self.seconds:
                raise FetchStop('request_or_time_budget_reached')
            time.sleep(max(0, REQUEST_INTERVAL_SECONDS - (time.monotonic() - self.last_call)))
            self.calls += 1
            self.last_call = time.monotonic()
            try:
                response = self.session.get(BASE_URL + '/' + endpoint,
                    params={**params, 'apikey': self.key}, timeout=(10, 40), allow_redirects=False)
            except requests.RequestException:
                self.log.append({'request_id': request_id, 'status': 'network_error', 'attempt': attempt + 1})
                if attempt == 2:
                    raise RuntimeError('network_error') from None
                continue
            status = response.status_code
            self.log.append({'request_id': request_id, 'status': status, 'attempt': attempt + 1})
            if status in [401, 403, 429]:
                raise FetchStop('http_' + str(status))
            if status >= 500 and attempt < 2:
                time.sleep(2 ** attempt)
                continue
            if status != 200:
                raise RuntimeError('http_' + str(status))
            try:
                data = response.json()
            except ValueError:
                raise RuntimeError('invalid_json') from None
            if not isinstance(data, list) or any(not isinstance(row, dict) for row in data):
                # Vendor error messages can echo URLs/keys: never print or archive them.
                raise RuntimeError('unexpected_payload_or_vendor_error')
            if any('Error Message' in row or 'error' in row for row in data):
                raise RuntimeError('vendor_error_payload')
            record = {'endpoint': endpoint, 'params': params, 'data': data,
                      'retrieved_utc': pd.Timestamp.now(tz='UTC').isoformat(),
                      'payload_sha256': hashlib.sha256(canonical_json(data).encode()).hexdigest()}
            write_json(path, record)
            return data, 'downloaded', request_id
        raise RuntimeError('retry_exhausted')


def cik_id(value):
    text = str(value).strip() if value is not None else ''
    if text.lower() in {'','none','null','nan','nat','<na>'}: return ''
    if text.endswith('.0') and text[:-2].isdigit(): text=text[:-2]
    if not text.isdigit(): return ''
    return text.lstrip('0')  # all-zero CIKs are missing, not a separate issuer

def cache_read(root, endpoint, params):
    rid = hashlib.sha256(canonical_json({'endpoint':endpoint,'params':params}).encode()).hexdigest()
    p = root/(rid+'.json')
    if not p.exists(): return None
    envelope=json.loads(p.read_text())
    if envelope.get('endpoint')!=endpoint or envelope.get('params')!=params:
        raise ValueError('Cache request mismatch')
    if hashlib.sha256(canonical_json(envelope['data']).encode()).hexdigest()!=envelope['payload_sha256']:
        raise ValueError('Cache checksum mismatch')
    if not isinstance(envelope['data'],list): raise ValueError('Cache schema mismatch')
    return envelope['data']

class RecoveryClient:
    def __init__(self, archive, old_roots):
        self.archive,self.old_roots=archive,old_roots
        self.rows,self.payloads,self.tasks=[],{},{}
        self.blocked=set();self.pause=None
    def get(self, endpoint, params, owner, purpose):
        rid=self.archive.request_id(endpoint,params)
        self.tasks[rid]={'request_id':rid,'endpoint':endpoint,'params_json':canonical_json(params)}
        base={'request_id':rid,'endpoint':endpoint,'owner':owner,'purpose':purpose}
        if rid in self.payloads:
            return self.payloads[rid],rid
        try:
            for root in self.old_roots+[self.archive.root]:
                records=cache_read(root,endpoint,params)
                if records is not None:
                    origin='cache';break
            else:
                if self.pause:
                    self.rows.append({**base,'status':'pending','reason':self.pause});return None,rid
                if endpoint in self.blocked:
                    self.rows.append({**base,'status':'endpoint_unavailable','reason':'prior_http_403'});return None,rid
                records,origin,_=self.archive.get(endpoint,params)
            self.payloads[rid]=records
            self.rows.append({**base,'status':'observed' if records else 'empty','rows':len(records),'origin':origin})
            return records,rid
        except FetchStop as error:
            reason=str(error)
            if reason=='http_403': self.blocked.add(endpoint)
            else:self.pause=reason
            self.rows.append({**base,'status':'endpoint_unavailable' if reason=='http_403' else 'pending','reason':reason})
        except (ValueError,TypeError,KeyError,OSError,RuntimeError) as error:
            self.rows.append({**base,'status':'error','reason':type(error).__name__})
        return None,rid

def identity_status(record, expected_cik):
    found=cik_id(record.get('cik'))
    if not found:return 'missing_cik'
    return 'issuer_cik_match_only' if found==expected_cik else 'different_issuer'

def inspect_prices(records, field, lo, hi, symbol):
    columns=['date',field,'reason']
    if not records:return pd.DataFrame(columns=columns)
    f=pd.DataFrame(records)
    dates=pd.to_datetime(f.get('date',pd.Series(None,index=f.index,dtype=object)),errors='coerce')
    values=pd.to_numeric(f.get(field,pd.Series(None,index=f.index,dtype=object)),errors='coerce')
    reasons=pd.Series('',index=f.index,dtype=object)
    reasons.loc[dates.isna()]='invalid_date'
    reasons.loc[values.isna()|~np.isfinite(values)|values.le(0)]+='|invalid_'+field
    reasons.loc[dates.notna() & (dates.lt(pd.Timestamp(lo))|dates.gt(pd.Timestamp(hi)))]+='|outside_window'
    reasons.loc[dates.notna() & dates.duplicated(keep=False)]+='|duplicate_date'
    if 'symbol' in f:reasons.loc[f.symbol.fillna('').ne(symbol)]+='|symbol_mismatch'
    return pd.DataFrame({'date':dates,field:values,'reason':reasons})

def rai_candidate(full, dividends, primary, sessions):
    """Diagnostic only: explicit adjusted-dividend field, never raw-dividend substitution."""
    report={'approved_for_training':False,'status':'insufficient_inputs'}
    if full.empty or not dividends:return pd.DataFrame(),report
    d=pd.DataFrame(dividends)
    if not {'date','adjDividend'}.issubset(d.columns):
        return pd.DataFrame(),{**report,'status':'missing_date_or_adjDividend'}
    d['date']=pd.to_datetime(d.date,errors='coerce')
    d['adjDividend']=pd.to_numeric(d.adjDividend,errors='coerce')
    if d.date.isna().any() or d.date.duplicated().any() or (~np.isfinite(d.adjDividend)|d.adjDividend.lt(0)).any():
        return pd.DataFrame(),{**report,'status':'invalid_or_duplicate_dividends'}
    f=full.loc[full.reason.eq(''),['date','close']].copy().sort_values('date')
    if f.empty or f.date.duplicated().any():return pd.DataFrame(),{**report,'status':'invalid_or_duplicate_prices'}
    f=f.set_index('date')
    # Only zero on dates absent from the supplied event ledger, explicitly conditional on completeness.
    f['dividend_assuming_complete_ledger']=d.set_index('date').adjDividend.reindex(f.index,fill_value=0)
    f['conditional_gross_return']=(f.close+f.dividend_assuming_complete_ledger)/f.close.shift(1)
    expected=sessions[(sessions>=f.index.min())&(sessions<=f.index.max())]
    positions=sessions.get_indexer(f.index)
    consecutive=np.r_[False,(np.diff(positions)==1)&(positions[1:]>=0)&(positions[:-1]>=0)]
    f.loc[~consecutive,'conditional_gross_return']=np.nan
    # Restart a diagnostic index after gaps; never bridge missing trading sessions.
    f['segment']=(~consecutive).cumsum()
    f['conditional_total_return_index']=f.groupby('segment').conditional_gross_return.transform(lambda s:s.fillna(1).cumprod()*100)
    events=d.loc[d.date.between(f.index.min(),f.index.max())]
    event_dates=pd.DatetimeIndex(events.date)
    report.update(status='UNAPPROVED_CONDITIONAL_RECONSTRUCTION',price_rows=len(f),
        missing_calendar_sessions=len(expected.difference(f.index)),event_rows_in_price_window=len(events),
        event_dates_without_prices=len(event_dates.difference(f.index)),
        largest_event_gap_days=int(events.date.sort_values().diff().dt.days.max()) if len(events)>1 else None,
        assumed_complete_dividend_ledger=True,split_basis_verified=False,
        warning='Zero between reported dividend events assumes a complete ledger; this is not approved data.')
    p=primary.loc[primary.reason.eq(''),['date','adjClose']].drop_duplicates('date').set_index('date').sort_index()
    if not p.empty:
        p['vendor_gross_return']=p.adjClose/p.adjClose.shift(1)
        pos=sessions.get_indexer(p.index)
        adjacent=np.r_[False,(np.diff(pos)==1)&(pos[1:]>=0)&(pos[:-1]>=0)]
        p.loc[~adjacent,'vendor_gross_return']=np.nan
        f=f.join(p['vendor_gross_return'])
        f['return_difference']=f.conditional_gross_return-f.vendor_gross_return
        dif=f.return_difference.dropna()
        event_overlap=f.loc[f.dividend_assuming_complete_ledger.gt(0),'return_difference'].dropna()
        report.update(overlap_return_rows=len(dif),overlap_dividend_rows=len(event_overlap),
            median_absolute_return_difference=float(dif.abs().median()) if len(dif) else None,
            maximum_absolute_return_difference=float(dif.abs().max()) if len(dif) else None)
    f['approved_for_training']=False
    return f.reset_index(),report

In [3]:
source_pins={}
for root in [SOURCE_RUN,V2_RUN]:
    manifest=json.loads((root/'output_sha256.json').read_text())
    for name,expected in manifest.items():
        p=root/name
        if p.parent!=root or not p.is_file() or digest_file(p)!=expected:raise ValueError('Source hash mismatch: '+name)
    source_pins[str(root)]=digest_file(root/'output_sha256.json')
calendar_hash=digest_file(VOO_CALENDAR)
intervals=pd.read_parquet(SOURCE_RUN/'candidate_membership_v3_UNVERIFIED.parquet')
coverage=pd.read_csv(SOURCE_RUN/'full_universe_price_coverage.csv')
source_report=json.loads((SOURCE_RUN/'reconciliation_report.json').read_text())
if source_report['structural_flags'] or source_report['count_warnings']:
    raise ValueError('Membership source is not the reviewed v3 result')
priority=(coverage.loc[coverage.year.ge(2016)].groupby('candidate_security_key').missing_sessions.sum().sort_values(ascending=False))
calendar=pd.read_parquet(VOO_CALENDAR,columns=['date'])
sessions=pd.DatetimeIndex(pd.to_datetime(calendar.date).unique()).sort_values()
archive=FMPArchive(NEW_CACHE,FMP_KEY,MAX_NEW_HTTP_ATTEMPTS,MAX_RUN_SECONDS,RUN_NETWORK)
client=RecoveryClient(archive,OLD_CACHES)
display(priority.head(15).rename('missing_member_sessions_2016_2019').to_frame())

,missing_member_sessions_2016_2019
candidate_security_key,
NBL,1006
STI,989
APC,905
RHT,887
LLL,877
FOXA,801
NFX,784
SCG,753
CA,716


## 1. Historical issuer evidence

These CIKs come from historical SEC filings. They anchor the identity investigation; they are not a global mapping across mergers, reorganizations or share classes. In particular, the FOXA row below refers to **Twenty-First Century Fox**, not the later Fox Corporation. Its membership-key split remains unresolved and no FOXA series is stitched automatically. Legacy Allergan Inc. remains separate from Actavis/Allergan plc.

API documentation checked for this revision: [FMP endpoint catalog](https://site.financialmodelingprep.com/developer/docs), [dividends](https://site.financialmodelingprep.com/developer/docs/stable/dividends-company), and [historical-price guide](https://site.financialmodelingprep.com/how-to/fmp-historical-price-apis-from-light-charts-to-dividendadjusted-analysis). `profile-cik` uses `cik`; `search-name` uses `query`; `delisted-companies` uses `page`/`limit`; dividends and splits use `symbol`. Entitlements remain plan-specific.

In [4]:
registry=pd.DataFrame([
 ('NBL','Noble Energy','72207','https://www.sec.gov/Archives/edgar/data/72207/000007220719000018/nbl-20181231x10k.htm'),
 ('STI','SunTrust Banks','750556','https://www.sec.gov/Archives/edgar/data/750556/000075055619000103/a123118form10-kxreport.htm'),
 ('APC','Anadarko Petroleum','773910','https://www.sec.gov/Archives/edgar/data/773910/000077391019000009/apc201810k-10k.htm'),
 ('RHT','Red Hat','1087423','https://www.sec.gov/Archives/edgar/data/1087423/000108742318000010/rht-10kq4fy18.htm'),
 ('LLL','L3 Technologies','1039101','https://www.sec.gov/Archives/edgar/data/1039101/000103910119000005/lll-123118x10k.htm'),
 ('FOXA','Twenty-First Century Fox','1308161','https://www.sec.gov/files/divisions/corpfin/cf-noaction/14a-8/2018/nathancummings082718-14a8.pdf'),
 ('NFX','Newfield Exploration','912750','https://www.sec.gov/Archives/edgar/data/912750/000091275018000004/nfx201710-k.htm'),
 ('SCG','SCANA','754737','https://www.sec.gov/Archives/edgar/data/754737/000075473718000170/a12312017scana10-ka.htm'),
 ('CA','CA Inc','356028','https://www.sec.gov/Archives/edgar/data/356028/000035602817000047/ca-20170331x10k.htm'),
 ('XL','XL Group','875159','https://www.sec.gov/Archives/edgar/data/875159/000087515918000015/xlgroup-10xk_2017.htm'),
 ('RAI','Reynolds American','1275283','https://www.sec.gov/Archives/edgar/data/1275283/000156459017001245/0001564590-17-001245-index.htm'),
 ('AGN_LEGACY_INC','Allergan Inc','850693','https://www.sec.gov/Archives/edgar/data/850693/000085069312000003/agn10-k.htm'),
],columns=['candidate_security_key','historical_name','expected_cik','evidence_url'])
registry['historical_ticker']=registry.candidate_security_key.replace({'AGN_LEGACY_INC':'AGN'})
registry['vendor_binding_approved']=False
registry.to_csv(RUN_DIR/'historical_issuer_registry.csv',index=False)
missing_priority=set(priority.head(10).index)-set(registry.candidate_security_key)
if missing_priority:raise ValueError('Priority universe changed; registry review required: '+str(sorted(missing_priority)))
display(registry[['candidate_security_key','historical_name','expected_cik']])

,candidate_security_key,historical_name,expected_cik
0,NBL,Noble Energy,72207
1,STI,SunTrust Banks,750556
2,APC,Anadarko Petroleum,773910
3,RHT,Red Hat,1087423
4,LLL,L3 Technologies,1039101
5,FOXA,Twenty-First Century Fox,1308161
6,NFX,Newfield Exploration,912750
7,SCG,SCANA,754737
8,CA,CA Inc,356028
9,XL,XL Group,875159


## 2. Discover and evaluate candidate vendor identities

A 403 blocks further calls to that endpoint during this run, while other endpoints can continue. A 401 or 429 pauses all new network calls. Successful caches remain readable. The delisted directory is paginated until empty, repeated content, error, or the explicit page cap; hitting the cap is **not** reported as exhaustive coverage.

Name search and delisted records produce leads only. Extra symbols are followed only when a profile supplies the expected nonzero CIK. Even then, retrieved prices stay unapproved because issuer identity alone cannot establish security continuity.

In [5]:
identity_rows=[]
metadata=[]
leads={r.candidate_security_key:set() for r in registry.itertuples()}
profile_matches={r.candidate_security_key:set() for r in registry.itertuples()}
def capture(owner,endpoint,rid,records,expected):
    for rec in records or []:
        # Whitelist identity metadata; exclude unrelated fields and any credential-bearing URLs.
        out={k:rec.get(k) for k in ['symbol','name','companyName','cik','exchange','exchangeShortName','ipoDate','delistedDate','isEtf','isFund','isActivelyTrading']}
        metadata.append({'owner':owner,'endpoint':endpoint,'request_id':rid,**out})
        identity_rows.append({'owner':owner,'endpoint':endpoint,'request_id':rid,'returned_symbol':rec.get('symbol'),
            'returned_name':rec.get('companyName',rec.get('name')),'expected_cik':expected,
            'returned_cik':cik_id(rec.get('cik')),'identity_result':identity_status(rec,expected),
            'approved_binding':False})
        sym=str(rec.get('symbol') or '').strip()
        if endpoint in ['profile','profile-cik'] and sym and identity_status(rec,expected)=='issuer_cik_match_only':
            if not rec.get('isEtf',False) and not rec.get('isFund',False):profile_matches[owner].add(sym)
for row in registry.itertuples():
    key=row.candidate_security_key
    for endpoint,params in [('profile-cik',{'cik':row.expected_cik}),
                            ('profile',{'symbol':row.historical_ticker}),
                            ('search-name',{'query':row.historical_name,'limit':50})]:
        records,rid=client.get(endpoint,params,key,'issuer_discovery')
        capture(key,endpoint,rid,records,row.expected_cik)
        if endpoint=='search-name':
            for rec in records or []:
                sym=str(rec.get('symbol') or '').strip()
                if sym:leads[key].add(sym)
    print(key,': CIK profile candidates',sorted(profile_matches[key]))
# Limited directory discovery. No claim that the directory spans all historical delistings.
directory_report={'pages_requested':0,'stop_reason':'page_cap','complete_response_pagination':False}
page_hashes=set()
def name_tokens(name):
    return set(re.findall(r'[a-z0-9]+',str(name).lower()))-{'inc','corp','corporation','company','co','plc','ltd','limited'}
for page in range(MAX_DELISTED_PAGES):
    records,rid=client.get('delisted-companies',{'page':page,'limit':100},'DIRECTORY','bounded_delisted_discovery')
    directory_report['pages_requested']+=1
    if records is None:
        directory_report['stop_reason']='request_unavailable';break
    if not records:
        directory_report.update(stop_reason='empty_page',complete_response_pagination=True);break
    signature=hashlib.sha256(canonical_json(records).encode()).hexdigest()
    if signature in page_hashes:
        directory_report['stop_reason']='repeated_page';break
    page_hashes.add(signature)
    for row in registry.itertuples():
        tokens=name_tokens(row.historical_name)
        matching=[rec for rec in records if rec.get('symbol')==row.historical_ticker or
                  (tokens and tokens.issubset(name_tokens(rec.get('companyName',rec.get('name','')))))]
        capture(row.candidate_security_key,'delisted-companies',rid,matching,row.expected_cik)
        for rec in matching:
            sym=str(rec.get('symbol') or '').strip()
            if sym:leads[row.candidate_security_key].add(sym)
# Resolve a bounded number of leads to profiles; names alone never authorize following an alias.
lead_limits=[]
for row in registry.itertuples():
    key=row.candidate_security_key
    candidates=sorted(leads[key]-profile_matches[key]-{row.historical_ticker})
    lead_limits.append({'owner':key,'profile_leads':len(candidates),'profile_leads_probed':min(len(candidates),MAX_PROFILE_LEADS_PER_ISSUER),
                        'truncated':len(candidates)>MAX_PROFILE_LEADS_PER_ISSUER})
    for symbol in candidates[:MAX_PROFILE_LEADS_PER_ISSUER]:
        records,rid=client.get('profile',{'symbol':symbol},key,'evaluate_discovered_symbol')
        capture(key,'profile',rid,records,row.expected_cik)
bindings=[]
for row in registry.itertuples():
    key=row.candidate_security_key
    bindings.append({'owner':key,'query_symbol':row.historical_ticker,'basis':'original_ticker_diagnostic_only','expected_cik':row.expected_cik,'approved':False})
    extra=sorted(profile_matches[key]-{row.historical_ticker})
    for symbol in extra[:MAX_EXTRA_SYMBOLS_PER_ISSUER]:
        bindings.append({'owner':key,'query_symbol':symbol,'basis':'profile_cik_match_issuer_only','expected_cik':row.expected_cik,'approved':False})
    lead_limits.append({'owner':key,'extra_symbol_candidates':len(extra),'extra_symbols_followed':min(len(extra),MAX_EXTRA_SYMBOLS_PER_ISSUER),
                        'truncated':len(extra)>MAX_EXTRA_SYMBOLS_PER_ISSUER})
pd.DataFrame(metadata).to_csv(RUN_DIR/'vendor_identity_metadata.csv',index=False)
pd.DataFrame(identity_rows).to_csv(RUN_DIR/'issuer_match_audit.csv',index=False)
pd.DataFrame(lead_limits).to_csv(RUN_DIR/'discovery_limits.csv',index=False)
pd.DataFrame(bindings).to_csv(RUN_DIR/'candidate_vendor_bindings_UNAPPROVED.csv',index=False)
display(pd.DataFrame(bindings))
print('Directory scan:',directory_report)

NBL : CIK profile candidates []
STI : CIK profile candidates []
APC : CIK profile candidates []
RHT : CIK profile candidates []
LLL : CIK profile candidates []
FOXA : CIK profile candidates []
NFX : CIK profile candidates []
SCG : CIK profile candidates []
CA : CIK profile candidates []
XL : CIK profile candidates []
RAI : CIK profile candidates ['RAI']
AGN_LEGACY_INC : CIK profile candidates []


,owner,query_symbol,basis,expected_cik,approved
0,NBL,NBL,original_ticker_diagnostic_only,72207,False
1,STI,STI,original_ticker_diagnostic_only,750556,False
2,APC,APC,original_ticker_diagnostic_only,773910,False
3,RHT,RHT,original_ticker_diagnostic_only,1087423,False
4,LLL,LLL,original_ticker_diagnostic_only,1039101,False
5,FOXA,FOXA,original_ticker_diagnostic_only,1308161,False
6,NFX,NFX,original_ticker_diagnostic_only,912750,False
7,SCG,SCG,original_ticker_diagnostic_only,754737,False
8,CA,CA,original_ticker_diagnostic_only,356028,False
9,XL,XL,original_ticker_diagnostic_only,875159,False


Directory scan: {'pages_requested': 20, 'stop_reason': 'page_cap', 'complete_response_pagination': False}


## 3. Retrieve candidate price and financial records

For each priority issuer, probe its original ticker and at most three extra CIK-matching vendor symbols. Original-ticker results remain diagnostic even if today's profile belongs to someone else. Existing empty responses are reused. Annual price windows span candidate membership plus two years of warm-up and one year after membership, bounded to 2009–2021.

Both dividend-adjusted and full EOD are inspected with their own schemas. Full EOD `close` is never counted as dividend-adjusted coverage. All raw candidate records are exported in a compressed file with request IDs for review; no series is spliced into the panel.

Statement CIK matches and historical-period coverage are counted separately. Original financial vintages, next-session timing, and reorganization boundaries still require N2 review.

In [6]:
price_summary,statement_summary=[],[]
rai_full_parts,rai_primary_parts=[],[]
# Keep a compact candidate inventory by date, without promoting it into full-universe coverage.
candidate_dates=[]
with gzip.open(RUN_DIR/'recovery_raw_records.jsonl.gz','wt',encoding='utf-8') as out:
    written=set()
    def archive_public(records,rid,endpoint):
        if records is None or rid in written:return
        written.add(rid)
        if endpoint.startswith('historical-price'):
            fields={'symbol','date','open','high','low','close','adjClose','volume','unadjustedVolume','change','changePercent'}
        else:
            fields={'symbol','date','cik','period','fiscalYear','calendarYear','acceptedDate','filingDate','reportedCurrency',
                    'revenue','netIncome','totalAssets','totalLiabilities','totalStockholdersEquity','operatingCashFlow','freeCashFlow'}
        for idx,rec in enumerate(records):
            out.write(canonical_json({'request_id':rid,'row_offset':idx,'endpoint':endpoint,'record':{k:v for k,v in rec.items() if k in fields}})+'\n')
    for binding in bindings:
        owner,symbol,expected=binding['owner'],binding['query_symbol'],binding['expected_cik']
        spans=intervals.loc[intervals.candidate_security_key.eq(owner)]
        if spans.empty:raise ValueError('Missing candidate membership for '+owner)
        first=max(2009,int(spans.member_from.min().year)-2)
        last=min(2021,int(spans.member_to.max().year)+1)
        for year in range(first,last+1):
            params={'symbol':symbol,'from':f'{year}-01-01','to':f'{year}-12-31'}
            member_sessions=sessions[sessions.year==year]
            member_mask=np.zeros(len(member_sessions),dtype=bool)
            for span in spans.itertuples():member_mask|=(member_sessions>=span.member_from)&(member_sessions<span.member_to)
            member_sessions=member_sessions[member_mask]
            for endpoint,field in [('historical-price-eod/dividend-adjusted','adjClose'),('historical-price-eod/full','close')]:
                records,rid=client.get(endpoint,params,owner,'price_recovery_candidate')
                archive_public(records,rid,endpoint)
                inspected=inspect_prices(records,field,params['from'],params['to'],symbol)
                valid=inspected.loc[inspected.reason.eq('')]
                dates=pd.DatetimeIndex(valid.date.unique())
                price_summary.append({'owner':owner,'query_symbol':symbol,'year':year,'endpoint':endpoint,'request_id':rid,
                    'field':field,'response_available':records is not None,'returned_rows':len(records or []),
                    'schema_has_required_fields':bool(records) and {'date',field}.issubset(pd.DataFrame(records).columns),
                    'valid_numeric_rows':len(valid),'expected_candidate_member_sessions':len(member_sessions),
                    'candidate_member_date_matches_UNAPPROVED':len(member_sessions.intersection(dates)),
                    'binding_basis':binding['basis'],'approved_recovered_rows':0})
                for date in dates:
                    candidate_dates.append({'owner':owner,'query_symbol':symbol,'date':date,'field':field,'request_id':rid,'approved':False})
                if owner=='RAI' and symbol=='RAI':
                    (rai_primary_parts if field=='adjClose' else rai_full_parts).append(inspected)
        for kind,endpoint in [('income','income-statement'),('balance','balance-sheet-statement'),('cashflow','cash-flow-statement')]:
            records,rid=client.get(endpoint,{'symbol':symbol,'period':'quarter','limit':1000},owner,'statement_recovery_candidate')
            archive_public(records,rid,endpoint)
            f=pd.DataFrame(records or [])
            counts={'issuer_cik_match_only':0,'different_issuer':0,'missing_cik':0}
            for rec in records or []:counts[identity_status(rec,expected)]+=1
            in_window=0
            if 'date' in f:
                dt=pd.to_datetime(f.date,errors='coerce')
                in_window=int(dt.between(pd.Timestamp('2009-01-01'),pd.Timestamp('2019-12-31')).sum())
            statement_summary.append({'owner':owner,'query_symbol':symbol,'kind':kind,'request_id':rid,
                'response_available':records is not None,'rows':len(f),'period_rows_2009_2019':in_window,**counts,'approved_rows':0})
        print('Reviewed:',owner,'via',symbol,'HTTP attempts:',archive.calls,'pause:',client.pause)
price_summary_df=pd.DataFrame(price_summary)
price_summary_df.to_csv(RUN_DIR/'candidate_price_recovery_summary.csv',index=False)
pd.DataFrame(statement_summary).to_csv(RUN_DIR/'candidate_statement_recovery_summary.csv',index=False)
pd.DataFrame(candidate_dates,columns=['owner','query_symbol','date','field','request_id','approved']).to_parquet(RUN_DIR/'candidate_price_dates_UNAPPROVED.parquet',index=False)
display(price_summary_df.groupby(['owner','field']).agg(valid_numeric_rows=('valid_numeric_rows','sum'),candidate_member_date_matches_UNAPPROVED=('candidate_member_date_matches_UNAPPROVED','sum')))
print('Counts across multiple candidate symbols may overlap. They are not net recovered universe coverage.')

Reviewed: NBL via NBL HTTP attempts: 78 pause: None
Reviewed: STI via STI HTTP attempts: 90 pause: None
Reviewed: APC via APC HTTP attempts: 102 pause: None
Reviewed: RHT via RHT HTTP attempts: 114 pause: None
Reviewed: LLL via LLL HTTP attempts: 126 pause: None
Reviewed: FOXA via FOXA HTTP attempts: 139 pause: None
Reviewed: NFX via NFX HTTP attempts: 151 pause: None
Reviewed: SCG via SCG HTTP attempts: 162 pause: None
Reviewed: CA via CA HTTP attempts: 173 pause: None
Reviewed: XL via XL HTTP attempts: 184 pause: None
Reviewed: RAI via RAI HTTP attempts: 185 pause: None
Reviewed: AGN_LEGACY_INC via AGN HTTP attempts: 193 pause: None


valid_numeric_rows  \
owner          field                          
AGN_LEGACY_INC adjClose                1952   
               close                   1952   
APC            adjClose                   0   
               close                      0   
CA             adjClose                   0   
               close                      0   
FOXA           adjClose                 710   
               close                    710   
LLL            adjClose                   0   
               close                      0   
NBL            adjClose                   7   
               close                      7   
NFX            adjClose                   0   
               close                      0   
RAI            adjClose                 477   
               close                   2154   
RHT            adjClose                   0   
               close                      0   
SCG            adjClose                   0   
               close                      0   
STI            adjClose                   0   
               close                      0   
XL             adjClose                   0   
               close                      0   

                         candidate_member_date_matches_UNAPPROVED  
owner          field                                               
AGN_LEGACY_INC adjClose                                      1056  
               close                                         1056  
APC            adjClose                                         0  
               close                                            0  
CA             adjClose                                         0  
               close                                            0  
FOXA           adjClose                                       205  
               close                                          205  
LLL            adjClose                                         0  
               close                                            0  
NBL            adjClose                                         0  
               close                                            0  
NFX            adjClose                                         0  
               close                                            0  
RAI            adjClose                                       477  
               close                                         1650  
RHT            adjClose                                         0  
               close                                            0  
SCG            adjClose                                         0  
               close                                            0  
STI            adjClose                                         0  
               close                                            0  
XL             adjClose                                         0  
               close                                            0

Counts across multiple candidate symbols may overlap. They are not net recovered universe coverage.


## 4. RAI: conditional reconstruction, not imputation

If FMP supplies `adjDividend`, this calculates the diagnostic daily gross return `(close[t] + adjDividend[t]) / close[t-1]`. It assumes closing prices and adjusted dividends share a split-adjustment basis, and assumes the supplied dividend ledger is complete. Both assumptions remain unverified. Missing adjusted-dividend fields are **not** replaced by raw dividends. Gaps in trading sessions break the diagnostic index into separate segments.

Dates absent from the supplied dividend ledger are assigned zero distributions **only within this explicitly conditional diagnostic**. This is not permission to impute missing outcomes or assert a complete dividend history. The notebook reports dividend coverage and compares available daily returns with valid primary adjusted prices. Agreement can support further investigation; it cannot by itself establish missing-event completeness, terminal merger proceeds or the correct security.

In [7]:
rai_full=pd.concat(rai_full_parts,ignore_index=True) if rai_full_parts else pd.DataFrame(columns=['date','close','reason'])
rai_primary=pd.concat(rai_primary_parts,ignore_index=True) if rai_primary_parts else pd.DataFrame(columns=['date','adjClose','reason'])
rai_dividends,dividend_rid=client.get('dividends',{'symbol':'RAI'},'RAI','reconstruction_evidence')
rai_splits,split_rid=client.get('splits',{'symbol':'RAI'},'RAI','reconstruction_evidence')
write_json(RUN_DIR/'RAI_dividends_raw.json',rai_dividends if rai_dividends is not None else [])
write_json(RUN_DIR/'RAI_splits_raw.json',rai_splits if rai_splits is not None else [])
rai_df,rai_report=rai_candidate(rai_full,rai_dividends,rai_primary,sessions)
rai_report.update(dividend_request_id=dividend_rid,split_request_id=split_rid,
                  dividends_response_available=rai_dividends is not None,splits_response_available=rai_splits is not None,
                  dividend_response_rows=len(rai_dividends or []),split_response_rows=len(rai_splits or []))
rai_df.to_csv(RUN_DIR/'RAI_CONDITIONAL_UNAPPROVED_returns.csv',index=False)
write_json(RUN_DIR/'RAI_reconstruction_diagnostics.json',rai_report)
print(json.dumps(rai_report,indent=2))

/var/folders/l3/hj4t10p12gs0m_nkjpjr9kqr0000gp/T/ipykernel_48575/3072193734.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rai_full=pd.concat(rai_full_parts,ignore_index=True) if rai_full_parts else pd.DataFrame(columns=['date','close','reason'])
/var/folders/l3/hj4t10p12gs0m_nkjpjr9kqr0000gp/T/ipykernel_48575/3072193734.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rai_primary=pd.concat(rai_primary_parts,ignore_index=True) if rai_primary_parts else pd.DataFrame(columns=['date','adjCl

{
  "approved_for_training": false,
  "status": "insufficient_inputs",
  "dividend_request_id": "4ee8decf35426f4b6574948f76dc43a42f9eabf217c3ba87c67d4b6be228c832",
  "split_request_id": "5d2955ce86aebb96adeb32f578828e4a43f228e4840015273d0d2930eb778b60",
  "dividends_response_available": true,
  "splits_response_available": true,
  "dividend_response_rows": 0,
  "split_response_rows": 3
}


## 5. Review package and decision boundary

The final report distinguishes successfully retrieved diagnostic prices from prices approved for use. The latter remains zero in this notebook: human review of source identity, share class, adjustment basis and corporate actions is still required. Broader historical coverage is not approved merely because an endpoint returned more rows.

Send this executed notebook and the review ZIP. Do not rerun N2/N3 against these candidate files yet. Existing validation periods and the sealed holdout remain unchanged. A green code check here means the notebook can execute and report failures honestly; it does not mean the expanded training dataset has passed its data-quality gates.

In [8]:
pd.DataFrame(client.rows).to_csv(RUN_DIR/'request_observations_v4.csv',index=False)
pd.DataFrame(client.tasks.values()).to_csv(RUN_DIR/'request_plan_v4.csv',index=False)
pd.DataFrame(archive.log).to_csv(RUN_DIR/'http_attempts.csv',index=False)
request_status=pd.DataFrame(client.rows).status.value_counts().to_dict()
report={'revision':'nb01_history_v4','run_id':RUN_ID,'status':'RECOVERY_CANDIDATES_REQUIRE_REVIEW',
 'source_manifest_hashes':source_pins,'voo_calendar_sha256':calendar_hash,
 'priority_issuers':len(registry),'new_http_attempts':archive.calls,'pause_reason':client.pause,
 'blocked_endpoints':sorted(client.blocked),'request_status_counts':request_status,
 'delisted_directory':directory_report,'candidate_bindings':len(bindings),
 'raw_price_summary_rows':len(price_summary),'statement_summary_rows':len(statement_summary),
 'approved_recovered_price_rows':0,'safe_to_replace_notebook2_inputs':False,'training_authorized':False,
 'rai_status':rai_report['status'],
 'scope':'Top historical price gaps plus RAI and legacy Allergan; not full API exhaustion'}
for root,expected in source_pins.items():
    root=Path(root)
    if digest_file(root/'output_sha256.json')!=expected:raise ValueError('Source manifest changed')
    for name,h in json.loads((root/'output_sha256.json').read_text()).items():
        if digest_file(root/name)!=h:raise ValueError('Source file changed: '+name)
if digest_file(VOO_CALENDAR)!=calendar_hash:raise ValueError('Calendar changed')
write_json(RUN_DIR/'recovery_report.json',report)
write_json(RUN_DIR/'output_sha256.json',{p.name:digest_file(p) for p in RUN_DIR.iterdir() if p.is_file()})
zip_path=RUN_DIR.parent/f'review_bundle_nb01_history_v4_{RUN_ID}.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN_DIR.iterdir()):
        if p.is_file():z.write(p,p.name)
print(json.dumps(report,indent=2))
print('Share review ZIP:',zip_path)

{
  "revision": "nb01_history_v4",
  "run_id": "20261020T142058312830Z",
  "status": "RECOVERY_CANDIDATES_REQUIRE_REVIEW",
  "source_manifest_hashes": {
    "/Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb01_history_v3/20261020T140634856943Z": "6cabf5c8ad0f86b16657d007c7a4b364facc0289dc6ade9ea6983accdf30b516",
    "/Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb01_history_v2/20261019T230406622432Z": "d9533942ef1e736deb238188f75da25d0b01d4a62bcbd4a0d82ae51ed6e83bb3"
  },
  "voo_calendar_sha256": "9fcb6f8a95f681abf821672d2eebd31dca1d97973bdf016f15b8ae0489e1f87c",
  "priority_issuers": 12,
  "new_http_attempts": 195,
  "pause_reason": null,
  "blocked_endpoints": [],
  "request_status_counts": {
    "empty": 268,
    "observed": 109
  },
  "delisted_directory": {
    "pages_requested": 20,
    "stop_reason": "page_cap",
    "complete_response_pagination": false
  },
  "candidate_bindings": 12,
  "raw_price_summary_rows": 274,
  "statement_summary_rows": 36,

## Delivery validation

Notebook schema and all code cells compile. Focused tests passed for missing/zero CIKs, issuer collisions, close-versus-adjusted-close schemas, the conditional dividend formula, missing adjusted-dividend refusal, and session-gap segmentation. A full offline run using your actual v2/v3 review bundles and a synthetic API transport completed 300 candidate price-window inspections across 13 fixture bindings, isolated a 403 endpoint failure, preserved an unrelated-issuer warning, and generated the review ZIP. Cache checksum corruption was rejected.

These tests validate implementation and reporting, not live API recovery or investment-data readiness. No live FMP request was made during delivery validation. Run locally, then share the executed notebook and v4 review ZIP.